In [ ]:
from typing import TypedDict
from langgraph.graph import END, StateGraph

class SimpleState(TypedDict):
    count: int


def increment(state: SimpleState) -> SimpleState: 
    return {
        "count": state["count"] + 1
    }

def should_continue(state):
    if(state["count"] < 5): 
        return "continue"
    else: 
        return "stop"
    
graph = StateGraph(SimpleState)

graph.add_node("increment", increment)

graph.set_entry_point("increment")

graph.add_conditional_edges(
    "increment", 
    should_continue, 
    {
        "continue": "increment", 
        "stop": END
    }
)

app = graph.compile()

state = {
    "count": 0
}

result = app.invoke(state)
print(result)

{'count': 5}


In [2]:
pip install langgraph


  Using cached langgraph-0.4.8-py3-none-any.whl.metadata (6.8 kB)
  Using cached langgraph_checkpoint-2.0.26-py3-none-any.whl.metadata (4.6 kB)
  Using cached langgraph_prebuilt-0.2.2-py3-none-any.whl.metadata (4.5 kB)
  Using cached langgraph_sdk-0.1.70-py3-none-any.whl.metadata (1.5 kB)
  Using cached xxhash-3.5.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached ormsgpack-1.10.0-cp311-cp311-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (43 kB)
Using cached langgraph-0.4.8-py3-none-any.whl (152 kB)
Using cached langgraph_checkpoint-2.0.26-py3-none-any.whl (44 kB)
Using cached langgraph_prebuilt-0.2.2-py3-none-any.whl (23 kB)
Using cached langgraph_sdk-0.1.70-py3-none-any.whl (49 kB)
Using cached xxhash-3.5.0-cp311-cp311-macosx_11_0_arm64.whl (30 kB)
Using cached ormsgpack-1.10.0-cp311-cp311-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl (376 kB)

[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To upda

In [16]:
from typing import TypedDict, List, Literal
from langgraph.graph import END, StateGraph

class ConversationState(TypedDict):
    messages: List[str]
    turn: Literal["agent_a", "agent_b"]


In [17]:
# from pydantic import BaseModel
# from typing import List, Literal
# from langchain import OpenAI
# from langgraph.graph import END, StateGraph
# class ConversationState(BaseModel):
#     messages: List[str]
#     turn: Literal["agent_a", "agent_b"]


In [18]:
def agent_a(state: ConversationState) -> ConversationState:
    new_message = "Agent A: What's the weather today?"
    return {
        "messages": state["messages"] + [new_message],
        "turn": "agent_b"
    }

def agent_b(state: ConversationState) -> ConversationState:
    new_message = "Agent B: It's sunny and 25°C."
    return {
        "messages": state["messages"] + [new_message],
        "turn": "agent_a"
    }


In [19]:
def controller(state: ConversationState) -> str:
    if len(state["messages"]) >= 6:  
        return "stop"
    return state["turn"]


In [20]:
graph = StateGraph(ConversationState)

graph.add_node("agent_a", agent_a)
graph.add_node("agent_b", agent_b)

graph.set_entry_point("agent_a")

graph.add_conditional_edges(
    "agent_a",
    controller,
    {"agent_a": "agent_a", "agent_b": "agent_b", "stop": END}
)

graph.add_conditional_edges(
    "agent_b",
    controller,
    {"agent_a": "agent_a", "agent_b": "agent_b", "stop": END}
)

app = graph.compile()


In [21]:
initial_state = {
    "messages": [],
    "turn": "agent_a"
}

result = app.invoke(initial_state)
print("\n".join(result["messages"]))


Agent A: What's the weather today?
Agent B: It's sunny and 25°C.
Agent A: What's the weather today?
Agent B: It's sunny and 25°C.
Agent A: What's the weather today?
Agent B: It's sunny and 25°C.


In [22]:
class NumberGraph:
    def __init__(self):
        self.graph = {}

    def add_edge(self, u, v):
        """Add a directed edge from u to v"""
        self.graph[u] = v

    def print_numbers(self, start):
        """Traverse and print from the starting node"""
        current = start
        while current in self.graph:
            print(current)
            current = self.graph[current]
        print(current)  # Print the last node (10)

# Create the graph and add edges from 1 to 10
graph = NumberGraph()
for i in range(1, 10):  # From 1 to 9
    graph.add_edge(i, i + 1)

# Print numbers from 1 to 10
graph.print_numbers(1)


1
2
3
4
5
6
7
8
9
10


In [24]:
class MultiplesOfTen:
    def __init__(self):
        self.graph={}
    def add_edge(self,u,v):
        """ADD DIRECTED EDGE FROM U TO V"""
        self.graph[u]=[]
        self.graph[u]=v
    def print_number(self,start):
        """Traverse and print from the starting node"""
        current=start
        while current in self.graph:
            print(current)
            current= self.graph[current]
        print(current)
# Create the graph and add edges from 1 to 10
graph= MultiplesOfTen()
for i in range(1,10):
    graph.add_edge(i,i*10)
print(graph.print_number(1))   

1
10
None


In [25]:
class MultiplesOfTen:
    def __init__(self):
        self.graph = {}

    def add_edge(self, u, v):
        """ADD DIRECTED EDGE FROM U TO V"""
        self.graph[u] = v  # Just store the next node

    def print_number(self, start):
        """Traverse and print from the starting node"""
        current = start
        while current in self.graph:
            print(current)
            current = self.graph[current]
        print(current)  # Print the last node (which has no outgoing edge)

# Create the graph and add edges like: 1→10, 2→20, ..., 9→90
graph = MultiplesOfTen()
for i in range(1, 10):
    graph.add_edge(i, i * 10)

graph.print_number(1)


1
10


In [26]:
class MultiplesOfTen:
    def __init__(self):
        self.graph = {}

    def add_edge(self, u, v):
        """Add a directed edge from u to v"""
        self.graph[u] = v

    def generate_mermaid(self):
        """Generate Mermaid diagram string from the graph"""
        mermaid_lines = ["graph TD"]
        for u, v in self.graph.items():
            mermaid_lines.append(f"    {u} --> {v}")
        return "\n".join(mermaid_lines)

# Create the graph and add edges: 1→10, 2→20, ..., 9→90
graph = MultiplesOfTen()
for i in range(1, 10):
    graph.add_edge(i, i * 10)

# Generate and print the Mermaid diagram
mermaid_diagram = graph.generate_mermaid()
print(mermaid_diagram)


graph TD
    1 --> 10
    2 --> 20
    3 --> 30
    4 --> 40
    5 --> 50
    6 --> 60
    7 --> 70
    8 --> 80
    9 --> 90


In [1]:
from langchain_openai import ChatOpenAI
from langchain.agents import tool, create_react_agent
import datetime
from langchain_community.tools import TavilySearchResults
from langchain import hub

llm = ChatOpenAI(model="gpt-4")

@tool
def get_system_time(format: str = "%Y-%m-%d %H:%M:%S"):
    """ Returns the current date and time in the specified format """

    current_time = datetime.datetime.now()
    formatted_time = current_time.strftime(format)
    return formatted_time

search_tool = TavilySearchResults(search_depth="basic")
react_prompt = hub.pull("hwchase17/react")


tools = [get_system_time, search_tool]

react_agent_runnable = create_react_agent(tools=tools, llm=llm, prompt=react_prompt)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [28]:
pip install langchain-openai==0.3.5

  Using cached langchain_openai-0.3.5-py3-none-any.whl.metadata (2.3 kB)
  Using cached langchain_core-0.3.65-py3-none-any.whl.metadata (5.8 kB)
  Using cached langsmith-0.3.45-py3-none-any.whl.metadata (15 kB)
  Using cached zstandard-0.23.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (3.0 kB)
Using cached langchain_openai-0.3.5-py3-none-any.whl (54 kB)
Using cached langchain_core-0.3.65-py3-none-any.whl (438 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 6.5 MB/s eta 0:00:00
Using cached langsmith-0.3.45-py3-none-any.whl (363 kB)
Using cached zstandard-0.23.0-cp311-cp311-macosx_11_0_arm64.whl (633 kB)
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.2.3
    Uninstalling langsmith-0.2.3:
      Successfully uninstalled langsmith-0.2.3
  Attempting uninstall: langchain-core
    Found existi